# 3. DevSecOps, Shared Responsibility & Threat Modeling

Two topics the SC-100 exam keeps asking about but that are easy to fumble:

1. **DevSecOps** — how security shifts *left* into the pipeline.
2. **Shared Responsibility** — who owns what between customer and Microsoft.
3. **STRIDE** — a simple way to threat-model a design (WAF Security pillar uses this).

All three are "architect-level" skills: you choose *where* and *how* controls are applied, not how to click them in a portal.

## Setup — run this once

This lab uses only the Python standard library (no extra dependencies).

1. Open a terminal in `security-certs/sc-100/01-best-practices/`.
2. Run `uv sync` to create the lab's `.venv`.
3. In VS Code, click the kernel picker (top-right of this notebook) and select **.venv (Python)**.
4. If the kernel doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

Then run the cell below to confirm Python is working.

In [ ]:
import sys, platform
print('Python :', sys.version.split()[0])
print('Kernel :', sys.executable)
print('OS     :', platform.system(), platform.release())
print('Ready to design secure architectures ✅')

## Part 1 — Shared Responsibility Model

The exam loves to ask *"who is responsible for X?"*. The model changes by service type. This is Microsoft's
current responsibility matrix ([Shared responsibility in the cloud](https://learn.microsoft.com/azure/security/fundamentals/shared-responsibility)):

```
                              On-prem   IaaS      PaaS      SaaS
Customer data                 You       You       You       You
Configurations & settings     You       You       You       You
Identities & users            You       You       You       You
Client devices                You       You       You       Shared
Applications                  You       You       Shared    Shared
Network controls              You       You       Shared    MSFT
Operating system              You       You       MSFT      MSFT
Physical hosts                You       MSFT      MSFT      MSFT
Physical network              You       MSFT      MSFT      MSFT
Physical datacenter           You       MSFT      MSFT      MSFT
```

**Four things are ALWAYS yours**, no matter the service model: *data, configurations, identities/accounts, and endpoints*.

### Read the "Shared" cells carefully — that is where the marks are

| Shared cell | Microsoft's half | Your half |
|---|---|---|
| Applications (PaaS, SaaS) | Runs and patches the platform / the app | Your code, its configuration, who may use it |
| Network controls (PaaS) | Baseline platform network security | Firewall rules, private endpoints, public-access toggles |
| Client devices (SaaS) | Provides device-management capability | Enrolling, hardening, monitoring the endpoint |

> 📚 **Older wording**: some material shows a row called *"Identity and directory infrastructure"* (Shared in PaaS
> and SaaS) rather than *"Identities and users"* (always Customer). Same split, different vocabulary: the directory
> **service** is Microsoft's, your **accounts and their access** are always yours.
>
> ⚠️ Note the IaaS column: identity infrastructure and network controls in IaaS are **entirely yours**, not shared.
> A common (wrong) version of this table shifts the whole IaaS column one step toward Microsoft.

In [ ]:
# Tiny lookup — 'who owns this in each model?'
# Matches learn.microsoft.com/azure/security/fundamentals/shared-responsibility
MATRIX = {
    'Customer data':             {'onprem':'you','iaas':'you',      'paas':'you',      'saas':'you'},
    'Configurations & settings': {'onprem':'you','iaas':'you',      'paas':'you',      'saas':'you'},
    'Identities & users':        {'onprem':'you','iaas':'you',      'paas':'you',      'saas':'you'},
    'Client devices':            {'onprem':'you','iaas':'you',      'paas':'you',      'saas':'shared'},
    'Applications':              {'onprem':'you','iaas':'you',      'paas':'shared',   'saas':'shared'},
    'Network controls':          {'onprem':'you','iaas':'you',      'paas':'shared',   'saas':'microsoft'},
    'Operating system':          {'onprem':'you','iaas':'you',      'paas':'microsoft','saas':'microsoft'},
    'Physical hosts':            {'onprem':'you','iaas':'microsoft','paas':'microsoft','saas':'microsoft'},
    'Physical network':          {'onprem':'you','iaas':'microsoft','paas':'microsoft','saas':'microsoft'},
    'Physical datacenter':       {'onprem':'you','iaas':'microsoft','paas':'microsoft','saas':'microsoft'},
}

NOTES = {
    ('Applications', 'paas'):     'Microsoft patches App Service; your code and its config are still yours.',
    ('Applications', 'saas'):     'Microsoft runs Exchange Online; sharing settings and mail flow rules are yours.',
    ('Network controls', 'paas'): 'Microsoft secures the platform fabric; the "public network access" switch is yours.',
    ('Client devices', 'saas'):   'Intune is offered by Microsoft; actually enrolling and hardening devices is yours.',
}

def who_owns(layer, model):
    return MATRIX[layer][model]

examples = [
    ('Patching the guest OS of an Azure VM',                        'Operating system',      'iaas'),
    ('Patching the OS underneath Azure SQL Database',               'Operating system',      'paas'),
    ('A user permanently deletes mail in Exchange Online',          'Customer data',         'saas'),
    ('NSG / firewall rules on a VNet holding your VMs',             'Network controls',      'iaas'),
    ('Turning off public network access on a storage account',      'Network controls',      'paas'),
    ('Deciding who is a Global Administrator in Microsoft 365',     'Identities & users',    'saas'),
    ('Hardening the laptop an employee uses to reach Microsoft 365','Client devices',        'saas'),
]
for q, layer, model in examples:
    owner = who_owns(layer, model)
    print(f'Q: {q}')
    print(f'   ({model.upper()} · {layer}) -> {owner.upper()}')
    note = NOTES.get((layer, model))
    if note:
        print(f'   note: {note}')
    print()


## Part 2 — DevSecOps: shifting security left

Traditional dev: security gets involved at the end ("the security team blocks my release"). DevSecOps bakes checks into every pipeline stage so issues are found when they're cheapest to fix.

```
Plan  →  Code  →  Build  →  Test  →  Release  →  Deploy  →  Operate
  │       │        │         │         │          │          │
  Threat  Secret   SAST      DAST      IaC scan    Signing    Runtime
  model   scan     (code)    (app)     (Bicep/TF)  + attest   protection (Defender)
```

In [ ]:
# Bad → Best pipeline design
bad = [
    'Secrets stored in .env files checked into git',
    'No code scanning; PR review is the only gate',
    'Docker image pushed straight to production',
    'Deploy happens from a developer laptop',
    'No runtime monitoring of the app',
]

best = [
    'Secrets in Key Vault, retrieved at runtime via Managed Identity',
    'GitHub Advanced Security: CodeQL (SAST) + secret scanning on every PR',
    'Defender for Containers scans the image at build AND runtime',
    'Pipeline uses workload-identity federation (no stored PAT/SP secret)',
    'IaC scanned (PSRule / tfsec); production deploy requires signed artifact + approval',
    'Defender for Cloud + Defender for App Service monitor runtime',
]

print('❌ BAD pipeline')
for x in bad: print(f'  - {x}')
print('\n✅ BEST pipeline (DevSecOps)')
for x in best: print(f'  + {x}')

In [ ]:
# Scoring a pipeline — count how many DevSecOps controls are present
CONTROLS = {
    'secret_scanning':     True,
    'sast_codeql':         True,
    'dependency_scanning': True,
    'iac_scanning':        False,   # try flipping this
    'container_scanning':  True,
    'signed_artifacts':    False,
    'workload_identity':   True,
    'runtime_protection':  True,
}
weight = {k: 1 for k in CONTROLS}
weight['signed_artifacts'] = 2          # supply-chain attacks → high weight
weight['runtime_protection'] = 2

earned = sum(weight[k] for k, v in CONTROLS.items() if v)
total  = sum(weight.values())
print(f'DevSecOps score: {earned}/{total} ({earned*100//total}%)')
print('\nMissing controls:')
for k, v in CONTROLS.items():
    if not v: print(f'  - {k}  (weight {weight[k]})')

## Part 3 — STRIDE threat modeling

STRIDE is the simplest useful threat model: for each component ask *"can any of these happen?"*.

| Letter | Threat | Property it breaks | Typical Azure control |
|---|---|---|---|
| **S** | Spoofing identity | Authentication | Entra ID, MFA, Managed Identity |
| **T** | Tampering with data | Integrity | TLS, code signing, Azure Policy |
| **R** | Repudiation (deny the action) | Non-repudiation | Activity logs, Sentinel, immutable logs |
| **I** | Information disclosure | Confidentiality | Encryption, Private Endpoints, RBAC |
| **D** | Denial of Service | Availability | DDoS Protection, autoscale, WAF rate limit |
| **E** | Elevation of privilege | Authorization | Least privilege, PIM, Conditional Access |

In [ ]:
# Tiny threat model for a public web app → SQL DB
threat_model = [
    ('Browser → Front Door',  'S', 'Attacker spoofs user',         'Entra ID + MFA + Conditional Access'),
    ('Browser → Front Door',  'D', 'Volumetric DDoS',              'DDoS Protection Standard + WAF rate limit'),
    ('Front Door → App',      'T', 'Request tampered on the wire', 'HTTPS only + WAF + cert pinning'),
    ('App → SQL DB',          'I', 'Secret stolen → DB read',      'Managed Identity (no password) + Private Endpoint'),
    ('App → SQL DB',          'E', 'App service over-privileged',  'Least-privilege DB role, no db_owner'),
    ('SQL DB',                'R', 'Admin denies a data change',   'SQL auditing to immutable storage + Sentinel'),
]
print(f'{"Edge":<24} {"STRIDE":<7} {"Threat":<35} Control')
print('-' * 105)
for edge, letter, threat, control in threat_model:
    print(f'{edge:<24} {letter:<7} {threat:<35} {control}')

## Part 4 — Defense in Depth quick drill

Defense in Depth = "if one layer fails, the next layer still stops the attack." The exam rewards answers that add controls at multiple layers rather than one silver bullet.

In [ ]:
LAYERS = [
    ('Physical',     'Microsoft datacenter security'),
    ('Identity',     'MFA + Conditional Access + PIM'),
    ('Perimeter',    'DDoS Protection + Azure Firewall'),
    ('Network',      'NSGs, Private Endpoints, segmentation'),
    ('Compute',      'Endpoint protection, patching, attack surface reduction (ASR) rules'),
    ('Application',  'WAF, secure SDLC, managed identities'),
    ('Data',         'Encryption at rest + in transit, sensitivity labels, DLP'),
]

# Simulate an attacker — at each layer, probability the attack is stopped
p_blocked = [0.30, 0.80, 0.60, 0.70, 0.70, 0.60, 0.50]

p_reach = 1.0
print('Attacker journey (single-layer vs full stack):\n')
for (layer, ctrl), p in zip(LAYERS, p_blocked):
    p_reach *= (1 - p)
    print(f'  {layer:<11} {ctrl}')
    print(f'              layer stops {int(p*100)}%  →  chance attack still reaches data: {p_reach*100:5.2f}%')

print()
print('Key insight: no single layer gets anywhere near zero, but MULTIPLYING the misses drives the')
print('residual risk toward zero. Adding a mediocre 50% layer to a good stack still halves what gets through.')
print()
print('Honest caveat: this model assumes the layers fail INDEPENDENTLY. Real attacks correlate — a stolen')
print('admin credential can walk through identity, network and data controls at once, which is exactly why')
print('"assume breach" (segmentation + detection) sits alongside "verify explicitly" in Zero Trust.')

## Key takeaways

1. **Data, configurations, identities, endpoints** — always yours, in every service model. Everything else shifts with IaaS → PaaS → SaaS, and the "Shared" cells (Applications in PaaS/SaaS, Network controls in PaaS, Client devices in SaaS) are where exam questions concentrate.
2. **Shift security left**: secret scanning → SAST → SCA → DAST → IaC scan → signed artifacts → runtime protection.
3. **STRIDE** is the 30-second threat model Microsoft expects you to use in architecture reviews.
4. **Defense in Depth** beats any single silver-bullet control — the exam rewards multi-layer answers.

You've now completed Lab 1. Move on to **[Lab 2 — SecOps, Identity and Compliance](../../02-secops-identity-compliance/)**.